In [4]:
import torch
import os
import time
import psutil
import onnxruntime as ort
from transformers import BlipForConditionalGeneration
from onnxruntime.quantization import quantize_dynamic, QuantType

# --- Configuration (Using expanduser to avoid Hugging Face errors) ---
MODEL_DIR = os.path.expanduser("~/Desktop/video_captioning/blip_video_model_2")
ONNX_MODEL_PATH = "blip_vision_encoder.onnx"
QUANTIZED_MODEL_PATH = "blip_vision_encoder_int8.onnx"

CHANNELS = 3
IMAGE_SIZE = 224 

def export_and_quantize():
    print(f"1. Loading local model from: {MODEL_DIR}")
    if not os.path.exists(MODEL_DIR):
        raise FileNotFoundError(f"🚨 Path not found: {MODEL_DIR}")

    # Load model and isolate the vision encoder
    model = BlipForConditionalGeneration.from_pretrained(MODEL_DIR)
    vision_encoder = model.vision_model
    vision_encoder.eval()

    # Create dummy input and export with dynamic batching
    dummy_input = torch.randn(1, CHANNELS, IMAGE_SIZE, IMAGE_SIZE)
    print("2. Exporting to ONNX...")
    torch.onnx.export(
        vision_encoder, dummy_input, ONNX_MODEL_PATH,
        export_params=True, opset_version=14, do_constant_folding=True,
        input_names=['pixel_values'], output_names=['last_hidden_state'],
        dynamic_axes={'pixel_values': {0: 'batch'}, 'last_hidden_state': {0: 'batch'}}
    )
    
    print("3. Quantizing to INT8...")
    quantize_dynamic(
        model_input=ONNX_MODEL_PATH, 
        model_output=QUANTIZED_MODEL_PATH,
        weight_type=QuantType.QUInt8
    )
    print("✅ Export & Quantization Complete!\n")

def benchmark(model_path, batch_sizes=[1, 4, 8]):
    print(f"--- Benchmarking: {model_path} ---")
    session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])
    input_name = session.get_inputs()[0].name
    
    print(f"{'Batch Size':<12} | {'Throughput (Frames/sec)':<25} | {'Peak Memory (MB)'}")
    for batch in batch_sizes:
        dummy_batch = torch.randn(batch, CHANNELS, IMAGE_SIZE, IMAGE_SIZE).numpy()
        
        # Measure Memory
        process = psutil.Process(os.getpid())
        mem_before = process.memory_info().rss / (1024 * 1024)
        
        # Measure Speed
        start = time.perf_counter()
        for _ in range(5): # Run 5 times for average
            session.run(None, {input_name: dummy_batch})
        end = time.perf_counter()
        
        peak_mem = (process.memory_info().rss / (1024 * 1024)) - mem_before
        throughput = (batch * 5) / (end - start)
        
        print(f"{batch:<12} | {throughput:<25.2f} | {max(0, peak_mem):.2f}")
    print("\n")

if __name__ == "__main__":
    export_and_quantize()
    # Fulfills Task 1, Instructions 5 & 6
    benchmark(ONNX_MODEL_PATH) 
    benchmark(QUANTIZED_MODEL_PATH)

1. Loading local model from: /Users/ayraj/Desktop/video_captioning/blip_video_model_2


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 17405.13it/s]
The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
/var/folders/wy/j_9qstb57t158j11hdk2_ql40000gn/T/ipykernel_95939/23029789.py:30: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


2. Exporting to ONNX...


W0325 12:32:32.758499 95939 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0325 12:32:32.759202 95939 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0325 12:32:32.759589 95939 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0325 12:32:32.760163 95939 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `BlipVisionModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `BlipVisionModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/copyreg.py:101: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/Users/ayraj/Desktop/video_captioning/venv/lib/python3.10/site-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
  File "/Users/ayraj/Desktop/video_captioning/venv/lib/python3.10/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/Users/ayraj/Desktop/video_captioning/venv/lib/python3.10/site-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
  File "/Users/ayraj/Desktop/video_captioning/venv/lib/python3.10/site-packages/onnx/version_converter.py", line 39, in convert_version
 

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 27 of general pattern rewrite rules.


3. Quantizing to INT8...
✅ Export & Quantization Complete!

--- Benchmarking: blip_vision_encoder.onnx ---
Batch Size   | Throughput (Frames/sec)   | Peak Memory (MB)
1            | 15.37                     | 36.72
4            | 15.87                     | 120.28
8            | 15.35                     | 175.98


--- Benchmarking: blip_vision_encoder_int8.onnx ---
Batch Size   | Throughput (Frames/sec)   | Peak Memory (MB)
1            | 34.87                     | 0.61
4            | 37.13                     | 71.16
8            | 37.45                     | 146.91




In [19]:
import time
import cv2
import torch
import os
import numpy as np
import onnxruntime as ort
from transformers import BlipForConditionalGeneration, AutoProcessor
from transformers.modeling_outputs import BaseModelOutputWithPooling

# --- Configuration ---
BASE_PATH = "/Users/ayraj/Desktop/video_captioning"
MODEL_DIR = os.path.join(BASE_PATH, "blip_video_model_2")
ONNX_ENCODER_PATH = os.path.join(BASE_PATH, "project3", "blip_vision_encoder_int8.onnx")
VIDEO_PATH = os.path.join(BASE_PATH, "TrainValVideo", "video1.mp4")
NUM_FRAMES = 8 

def profile_pipeline():
    print("Loading models and processor...")
    processor = AutoProcessor.from_pretrained(MODEL_DIR)
    processor.image_processor.size = {"height": 224, "width": 224}
    
    pytorch_model = BlipForConditionalGeneration.from_pretrained(MODEL_DIR)
    pytorch_model.eval()
    
    onnx_session = ort.InferenceSession(ONNX_ENCODER_PATH, providers=['CPUExecutionProvider'])
    input_name = onnx_session.get_inputs()[0].name

    print(f"\n--- Starting Video Processing ---")
    t0 = time.perf_counter()
    
    # ==========================================
    # 1. ROBUST FRAME LOADING
    # ==========================================
    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        raise ValueError(f"❌ OpenCV could not open the video file at {VIDEO_PATH}. Is it a valid .mp4?")

    frames = []
    # Try to read frames sequentially to bypass bad metadata
    count = 0
    while len(frames) < NUM_FRAMES:
        ret, frame = cap.read()
        if not ret:
            break
        # Take every 5th frame to get some motion, or just the first 8 if the video is short
        if count % 5 == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (224, 224))
            frames.append(frame)
        count += 1
    cap.release()

    if len(frames) == 0:
        raise ValueError("❌ No frames were captured. The video file might be corrupted or empty.")
    
    # If we got fewer than NUM_FRAMES, just duplicate the last one to fill the batch
    while len(frames) < NUM_FRAMES:
        frames.append(frames[-1])

    print(f"✅ Successfully captured {len(frames)} frames.")
    
    inputs = processor(images=frames, return_tensors="pt")
    pixel_values = inputs.pixel_values.numpy()
    
    t1 = time.perf_counter()

    # ==========================================
    # 2. ENCODING (ONNX INT8)
    # ==========================================
    onnx_outputs = onnx_session.run(None, {input_name: pixel_values})
    image_embeds = torch.tensor(onnx_outputs[0])
    
    t2 = time.perf_counter()

    # ==========================================
    # 3. TEXT DECODING (PyTorch + Mocking)
    # ==========================================
    class MockVisionModel(torch.nn.Module):
        def __init__(self, precomputed_embeds):
            super().__init__()
            self.embeds = precomputed_embeds
            self.config = pytorch_model.config.vision_config # Pass config for stability
        def forward(self, *args, **kwargs):
            return BaseModelOutputWithPooling(
                last_hidden_state=self.embeds,
                pooler_output=None # BLIP Decoder usually ignores this
            )

    original_vision_model = pytorch_model.vision_model
    pytorch_model.vision_model = MockVisionModel(image_embeds)

    with torch.no_grad():
        # Using a minimal decoy
        dummy_pixels = torch.zeros(NUM_FRAMES, 3, 224, 224)
        out_ids = pytorch_model.generate(
            pixel_values=dummy_pixels, 
            max_length=30,
            num_beams=1 # Speed up for profiling
        )
    
    caption = processor.decode(out_ids[0], skip_special_tokens=True)
    pytorch_model.vision_model = original_vision_model
    
    t3 = time.perf_counter()
    total = t3 - t0

    print(f"\n[Generated Caption]: {caption}")
    print("\n--- Latency Profile (Mac M-Series) ---")
    print(f"1. Frame Loading: {t1-t0:.4f} sec ({((t1-t0)/total)*100:.1f}%)")
    print(f"2. ONNX Encoding: {t2-t1:.4f} sec ({((t2-t1)/total)*100:.1f}%)")
    print(f"3. Text Decoding: {t3-t2:.4f} sec ({((t3-t2)/total)*100:.1f}%)")
    print("-" * 38)
    print(f"Total Latency:    {total:.4f} sec")

if __name__ == "__main__":
    profile_pipeline()

Loading models and processor...


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 18467.31it/s]
The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



--- Starting Video Processing ---
✅ Successfully captured 8 frames.

[Generated Caption]: a man is cooking in a kitchen

--- Latency Profile (Mac M-Series) ---
1. Frame Loading: 0.0129 sec (0.7%)
2. ONNX Encoding: 0.2105 sec (12.0%)
3. Text Decoding: 1.5240 sec (87.2%)
--------------------------------------
Total Latency:    1.7474 sec
